In [1]:
import pandas as pd
import random
from pyfaidx import Fasta
from pybigtools import open

In [11]:
genome = Fasta(
    "../data/reference/hg38.fa"
)

bb = open(
    "../data/raw/ENCFF106WBX.bigBed"
)

In [12]:
WINDOW_SIZE = 1000

def extract_fixed_window(
    genome,
    chrom,
    start,
    peak_offset,
    window_size=WINDOW_SIZE
):

    summit = start + peak_offset

    half_window = window_size // 2

    window_start = summit - half_window
    window_end = summit + half_window

    sequence = genome[
        chrom
    ][
        window_start:window_end
    ].seq.upper()

    return sequence

In [13]:
def overlaps_peak(
    chrom,
    query_start,
    query_end,
    peak_intervals
):

    for peak_start, peak_end in peak_intervals[chrom]:

        if (
            query_start < peak_end
            and
            query_end > peak_start
        ):
            return True

    return False

In [14]:
positive_sequences = []

chromosomes = [
    "chr1",
    "chr2",
    "chr3"
]

for chrom in chromosomes:

    print(f"Processing {chrom}")

    chrom_size = bb.chroms()[chrom]

    records = list(
        bb.records(
            chrom,
            0,
            chrom_size
        )
    )

    for record in records:

        (
            start,
            end,
            name,
            score,
            strand,
            signalValue,
            pValue,
            qValue,
            peak
        ) = record

        sequence = extract_fixed_window(
            genome=genome,
            chrom=chrom,
            start=start,
            peak_offset=int(peak)
        )

        if len(sequence) != WINDOW_SIZE:
            continue

        positive_sequences.append({
            "chrom": chrom,
            "start": start,
            "end": end,
            "sequence": sequence,
            "label": 1
        })

Processing chr1
Processing chr2
Processing chr3


In [15]:
positive_df = pd.DataFrame(
    positive_sequences
)

print(positive_df.head())
print(len(positive_df))

  chrom   start     end                                           sequence  \
0  chr1  778479  779226  TTTACCTCAATCTCGCAGGCTGGAATGCTGTGGTGGGATCATAGCT...   
1  chr1  778479  779226  ACCCGCGCCTCTAACGTCGCCAACGGCCCACCTCTATGGTGTCGGC...   
2  chr1  817132  817914  AGCCACATACAGAAGACTGAAGCCAGACCCCTACCTTTCACCATAT...   
3  chr1  826649  828088  GACGGGGAAGCATCTCAGGGAGGACTGGCGTCTGCCGAATCCCAGG...   
4  chr1  826649  828088  GGAAACTAATTTGGAAATCAGAAAACCACTAAGGAATTTGGGAATT...   

   label  
0      1  
1      1  
2      1  
3      1  
4      1  
24708


In [16]:
peak_intervals = {}

for chrom in chromosomes:

    peak_intervals[chrom] = []

    chrom_size = bb.chroms()[chrom]

    records = list(
        bb.records(
            chrom,
            0,
            chrom_size
        )
    )

    for record in records:

        start = record[0]
        end = record[1]

        peak_intervals[chrom].append(
            (start, end)
        )

In [17]:
negative_sequences = []

while len(negative_sequences) < len(positive_sequences):

    chrom = random.choice(chromosomes)

    chrom_size = bb.chroms()[chrom]

    random_start = random.randint(
        0,
        chrom_size - WINDOW_SIZE
    )

    random_end = random_start + WINDOW_SIZE

    if overlaps_peak(
        chrom,
        random_start,
        random_end,
        peak_intervals
    ):
        continue

    sequence = genome[
        chrom
    ][
        random_start:random_end
    ].seq.upper()

    if len(sequence) != WINDOW_SIZE:
        continue

    negative_sequences.append({
        "chrom": chrom,
        "start": start,
        "end": end,
        "sequence": sequence,
        "label": 0
    })

In [18]:
negative_df = pd.DataFrame(
    negative_sequences
)

print(negative_df.head())
print(len(negative_df))

  chrom      start        end  \
0  chr2  198157266  198157656   
1  chr3  198157266  198157656   
2  chr3  198157266  198157656   
3  chr2  198157266  198157656   
4  chr1  198157266  198157656   

                                            sequence  label  
0  CTGTTTTCCTTTGGTTAAGCCATTATTTGTTGCTGCTGCTGTTGTT...      0  
1  TTTACCATCCTGCTAGTTCACTTATTAATGAGTTAAACAAATCTTC...      0  
2  GAGGATGACATCTGGGTCCCACTCTTTAGTCCAATTCACGAATTCA...      0  
3  ATCTAGCTGCCTTTATTTAAATGCCACAATTCATCTTATAATGGTA...      0  
4  AGCAACCTGGATAACTGGAGATCCTTTTTCTAAGTGAAGTAACTCA...      0  
24708


In [23]:
dataset_df = pd.concat(
    [positive_df, negative_df],
    ignore_index=True
)

dataset_df = dataset_df.sample(
    frac=1
).reset_index(drop=True)

print(dataset_df.head())

  chrom      start        end  \
0  chr1  198157266  198157656   
1  chr2   47272064   47272853   
2  chr2   91496906   91498112   
3  chr2  186589633  186590651   
4  chr1  198157266  198157656   

                                            sequence  label  
0  TATTTTTAACTTCCTGCTGGTTCATTAATTCACTGGTTAGTGAATT...      0  
1  CTGGAAGGGAAGGAAGGGTGGGGGTGGCCTGCAGAGGGTGTGGCGA...      1  
2  AGCTCTTCTACGAATAATGTTTTTGTATGTAAAGGAGAAGCAAAAA...      1  
3  CCTGCGAATCCTTTCTTTGACTACTAAGTTAAGCTTTACATACCTC...      1  
4  GTGTGTGTGTATATATATATGAATCTATATGAATTCACCATATATT...      0  


In [24]:
dataset_df.to_csv(
    "../data/processed/1kb-liver_accessibility_gc_coords.csv",
    index=False
)

In [ ]:
# dataset_df = pd.read_csv(
#     "../data/processed/1kb-liver_accessibility_gc_matched_coords.csv"
# )

In [20]:
bw = open(
    "../data/raw/liver_h3k27ac_foldchange.bigWig"
)

print(bw.chroms())

{'chr1': 248956422, 'chr10': 133797422, 'chr11': 135086622, 'chr11_KI270721v1_random': 100316, 'chr12': 133275309, 'chr13': 114364328, 'chr14': 107043718, 'chr14_GL000009v2_random': 201709, 'chr14_GL000194v1_random': 191469, 'chr14_GL000225v1_random': 211173, 'chr14_KI270722v1_random': 194050, 'chr14_KI270723v1_random': 38115, 'chr14_KI270724v1_random': 39555, 'chr14_KI270725v1_random': 172810, 'chr15': 101991189, 'chr15_KI270727v1_random': 448248, 'chr16': 90338345, 'chr16_KI270728v1_random': 1872759, 'chr17': 83257441, 'chr17_GL000205v2_random': 185591, 'chr17_KI270729v1_random': 280839, 'chr17_KI270730v1_random': 112551, 'chr18': 80373285, 'chr19': 58617616, 'chr1_KI270706v1_random': 175055, 'chr1_KI270707v1_random': 32032, 'chr1_KI270708v1_random': 127682, 'chr1_KI270709v1_random': 66860, 'chr1_KI270710v1_random': 40176, 'chr1_KI270711v1_random': 42210, 'chr1_KI270712v1_random': 176043, 'chr1_KI270713v1_random': 40745, 'chr1_KI270714v1_random': 41717, 'chr2': 242193529, 'chr20': 64

In [32]:
h3k27ac_signals = []

In [33]:
import numpy as np

for idx, row in dataset_df.iterrows():

    chrom = row["chrom"]

    start = row["start"]

    end = row["end"]

    values = bw.values(
        chrom,
        start,
        end
    )

    mean_signal = np.nanmean(values)

    if np.isnan(mean_signal):

        mean_signal = 0.0

    h3k27ac_signals.append(
        mean_signal
    )

    if idx % 1000 == 0:

        print(idx)

0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000
48000
49000


In [34]:
dataset_df["h3k27ac"] = h3k27ac_signals

In [35]:
dataset_df["h3k27ac"].describe()

count    49416.000000
mean         6.236938
std          7.401217
min          0.000000
25%          0.000000
50%          2.572277
75%         13.673236
max         71.265356
Name: h3k27ac, dtype: float64

In [36]:
positive_signals = dataset_df[
    dataset_df["label"] == 1
]["h3k27ac"]

negative_signals = dataset_df[
    dataset_df["label"] == 0
]["h3k27ac"]

print(positive_signals.mean())

print(negative_signals.mean())

7.403286138383175
5.070589861449868


In [37]:
dataset_df["h3k27ac"] = np.log1p(
    dataset_df["h3k27ac"]
)

mean = dataset_df["h3k27ac"].mean()

std = dataset_df["h3k27ac"].std()

dataset_df["h3k27ac"] = (
    dataset_df["h3k27ac"] - mean
) / std

dataset_df["h3k27ac"].describe()

count    4.941600e+04
mean     3.375423e-17
std      1.000000e+00
min     -1.102533e+00
25%     -1.102533e+00
50%     -4.739415e-02
75%      1.123450e+00
max      2.444707e+00
Name: h3k27ac, dtype: float64

In [38]:
dataset_df.to_csv(
    "../data/processed/1kb_liver_accessibility_multimodal.csv",
    index=False
)

In [2]:
dataset_df = pd.read_csv(
    "../data/processed/1kb_liver_accessibility_multimodal.csv"
)

In [3]:
from src.utils import reverse_complement

augmented_rows = []

for _, row in dataset_df.iterrows():

    augmented_rows.append(row)

    reverse_row = row.copy()

    reverse_row["sequence"] = reverse_complement(
        row["sequence"]
    )

    augmented_rows.append(reverse_row)

dataset_df = pd.DataFrame(
    augmented_rows
).reset_index(drop=True)

In [4]:
print(len(dataset_df))

98832


In [5]:
print(dataset_df.head())

  chrom      start        end  \
0  chr1  198157266  198157656   
1  chr1  198157266  198157656   
2  chr2   47272064   47272853   
3  chr2   47272064   47272853   
4  chr2   91496906   91498112   

                                            sequence  label   h3k27ac  
0  TATTTTTAACTTCCTGCTGGTTCATTAATTCACTGGTTAGTGAATT...      0  1.224731  
1  TTCCCTGTATCCACACACCAGTAATCCCCTCTGTCCACACACCAGT...      0  1.224731  
2  CTGGAAGGGAAGGAAGGGTGGGGGTGGCCTGCAGAGGGTGTGGCGA...      1 -0.021313  
3  TGCTCCCTTAACTCTATGTTCCCCTTCTCCTTTTCCCTTAGAGAAG...      1 -0.021313  
4  AGCTCTTCTACGAATAATGTTTTTGTATGTAAAGGAGAAGCAAAAA...      1 -0.895101  


In [6]:
dataset_df.to_csv(
    "../data/processed/1kb_liver_accessibility_multimodal_rc.csv",
    index=False
)